In [ ]:
import os, re, io, time, asyncio, threading, subprocess

MODEL = 'qwen2.5:32b'   # biggest that fits both T4s (smartest local). 'qwen2.5:14b' = faster, '7b' = fastest.

# ---- 1. System deps + Ollama + edge-tts ------------------------------------------------
subprocess.run('apt-get -qq update && apt-get -qq install -y zstd ffmpeg', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run('pip install -q edge-tts aiohttp faster-whisper', shell=True, check=True)

# ---- 2. Start Ollama, pull + pre-warm the model ----------------------------------------
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_ORIGINS='*',
           OLLAMA_KEEP_ALIVE='24h', OLLAMA_SCHED_SPREAD='1')
subprocess.Popen(['ollama', 'serve'], env=env)
time.sleep(6)
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Warming model into the GPUs...')
subprocess.run(['ollama', 'run', MODEL, 'Say ready.'], check=True)
print('Model warm:', MODEL)

# ---- 3. Voice server: POST /tts {text, voice, rate, pitch} -> WAV audio -----------------
import edge_tts
from aiohttp import web

# Whisper speech-to-text (understands Urdu/Roman-Urdu far better than Windows dictation).
from faster_whisper import WhisperModel
try:
    _whisper = WhisperModel('medium', device='cuda', compute_type='int8_float16')
    print('Whisper on GPU')
except Exception as _e:
    _whisper = WhisperModel('medium', device='cpu', compute_type='int8')
    print('Whisper on CPU (', _e, ')')

def _mp3_to_wav(mp3: bytes) -> bytes:
    # 24kHz mono 16-bit WAV so the desktop app plays it with no codec dependency.
    p = subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', 'pipe:0',
                        '-ar', '24000', '-ac', '1', '-f', 'wav', 'pipe:1'],
                       input=mp3, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return p.stdout if p.returncode == 0 and p.stdout else mp3  # fall back to mp3 if ffmpeg missing

async def _tts(request):
    try:
        d = await request.json()
    except Exception:
        return web.Response(status=400, text='bad json')
    text = (d.get('text') or '').strip()
    voice = d.get('voice') or 'ur-PK-UzmaNeural'
    rate = d.get('rate') or '+0%'
    pitch = d.get('pitch') or '+0Hz'
    if not re.fullmatch(r'[+-]\d{1,3}%', rate):
        rate = '+0%'
    if not re.fullmatch(r'[+-]\d{1,3}Hz', pitch):
        pitch = '+0Hz'
    if not text:
        return web.Response(status=400, text='no text')
    buf = io.BytesIO()
    try:
        async for chunk in edge_tts.Communicate(text, voice, rate=rate, pitch=pitch).stream():
            if chunk['type'] == 'audio':
                buf.write(chunk['data'])
    except Exception as e:
        return web.Response(status=502, text=f'tts failed: {e}')
    wav = _mp3_to_wav(buf.getvalue())
    ctype = 'audio/wav' if wav[:4] == b'RIFF' else 'audio/mpeg'
    return web.Response(body=wav, content_type=ctype)

async def _stt(request):
    data = await request.read()
    if len(data) < 2048:
        return web.json_response({'text': ''})
    lang = request.query.get('lang')
    if lang not in ('ur', 'en'):
        lang = None
    import tempfile, os
    tf = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    tf.write(data); tf.close()
    def _run():
        segs, _info = _whisper.transcribe(tf.name, language=lang, beam_size=5, vad_filter=True)
        return ''.join(s.text for s in segs).strip()
    try:
        text = await asyncio.get_event_loop().run_in_executor(None, _run)
    except Exception as e:
        return web.json_response({'text': '', 'error': str(e)}, status=502)
    finally:
        try: os.unlink(tf.name)
        except Exception: pass
    return web.json_response({'text': text})

def _run_tts_server():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    app = web.Application()
    app.router.add_post('/tts', _tts)
    app.router.add_post('/stt', _stt)
    app.router.add_get('/', lambda r: web.Response(text='ilham-tts ok'))
    runner = web.AppRunner(app); loop.run_until_complete(runner.setup())
    loop.run_until_complete(web.TCPSite(runner, '0.0.0.0', 8080).start())
    print('Voice server listening on :8080')
    loop.run_forever()

threading.Thread(target=_run_tts_server, daemon=True).start()
time.sleep(2)

# ---- 4. Two Cloudflare tunnels: one for the brain (11434), one for the voice (8080) ----
subprocess.run('curl -sL -o cloudflared '
               'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
               '&& chmod +x cloudflared', shell=True, check=True)

def start_tunnel(port):
    p = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://localhost:{port}', '--no-autoupdate'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in p.stdout:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0); break
    threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()  # drain logs
    return url

brain_url = start_tunnel(11434)
voice_url = start_tunnel(8080)

print('\n==================================================')
print('  Paste into Ilham Console on your PC:')
print('     /set ollamaurl ' + str(brain_url))
print('     /set model ' + MODEL)
print('     /set ttsurl ' + str(voice_url))
print('     /model      (check brain)')
print('     /tts        (check voice)')
print('     (mic STT uses the same voice URL automatically)')
print('  Keep this cell running. One Ilham at a time.')
print('==================================================\n')

while True:
    time.sleep(60)